In [ ]:
!pip install scikit-posthocs

In [ ]:
import pandas as pd # pandas
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import seaborn as sns
from scipy.stats import mannwhitneyu, wilcoxon, friedmanchisquare # mannwhitneyu and wilcoxon
import scikit_posthocs as sp
from sklearn.linear_model import Perceptron         # perceptron
from sklearn.neighbors import KNeighborsClassifier  # knn
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression # logistic regression
from sklearn.model_selection import cross_val_score, StratifiedKFold # cross-validation and kfold
from sklearn.pipeline import Pipeline # pipeline
from sklearn.preprocessing import MinMaxScaler # minmaxscaler

In [ ]:
# Importa dataset
dataset = pd.read_csv("https://raw.githubusercontent.com/joaovictorlopezpereira/Heart-Failure-Predictor/refs/heads/main/csvs/heart-treated-filtered.csv")

In [ ]:
# Imprime dataset para conferir corretude da importação
display(dataset)

In [ ]:
# Função que faz o K-fold Cross Validation
def run_validation(model, X, y, scoring_metric, k, n_runs):
    scores = []
    pipe = Pipeline([
        ("scaler", MinMaxScaler()),
        ("model", model)
    ])
    for seed in range(n_runs):
        cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        cv_score = cross_val_score(pipe, X, y, cv=cv, scoring=scoring_metric)
        scores.append(cv_score.mean())
    return np.array(scores) # Retorna o array de médias do experimento

In [ ]:
# Inicializa knn com 5 vizinhos
knn = KNeighborsClassifier(n_neighbors=7)

# Inicializa reglog com 5000 iterações
reglog = LogisticRegression(max_iter=5000)

# Inicializa perceptron max_iter = 2000
perceptron = Perceptron(max_iter=2000, tol=1e-3)

# Inicializa árvore de decisão
# arvdec = DecisionTreeClassifier(max_depth=5)


In [ ]:
# Inicializa X e y como "dataset sem a coluna target" e "coluna target"
X = dataset.drop(columns=["HeartDisease"])
y = dataset['HeartDisease']

# Inicializa k e n_runs
k = 10 # Número de folds do kfold
n_runs = 101 # Número de rodadas

In [ ]:
# Mede a acurácia dos modelos
knn_accuracy = run_validation(knn, X, y, 'accuracy', k, n_runs)
reglog_accuracy = run_validation(reglog, X, y, 'accuracy', k, n_runs)
perceptron_accuracy = run_validation(perceptron, X, y, 'accuracy', k, n_runs)
# arvdec_accuracy = run_validation(arvdec, X, y, 'accuracy', k, n_runs)

In [ ]:
# Mede o f1 dos modelos
knn_f1 = run_validation(knn, X, y, 'f1', k, n_runs)
reglog_f1 = run_validation(reglog, X, y, 'f1', k, n_runs)
perceptron_f1 = run_validation(perceptron, X, y, 'f1', k, n_runs)

In [ ]:
# Plota o histograma das acurácias do teste
plt.hist(knn_accuracy, bins=30, alpha=0.5, label="Acurácia do KNN")
plt.hist(reglog_accuracy, bins=30, alpha=0.5, label="Acurácia da Reg. Log.")
plt.hist(perceptron_accuracy, bins=30, alpha=0.5, label="Acurácia do Perceptron")
# plt.hist(arvdec_accuracy, bins=30, alpha=0.5, label="Acurácia da Árvore de Decisão")
plt.legend()
plt.show()


In [ ]:
# Plota o histograma dos f1-score do teste
plt.hist(knn_f1, bins=30, alpha=0.5, label="f1-score do KNN")
plt.hist(reglog_f1, bins=30, alpha=0.5, label="f1-score da Reg. Log.")
plt.hist(perceptron_f1, bins=30, alpha=0.5, label="f1-score do Perceptron")
plt.legend()
plt.show()


In [ ]:
def show_boxplot(score_1, score_2, name1, name2):
  # Criação de um dataset temporário
  df = pd.DataFrame({
      'Acurácia': np.concatenate([score_1, score_2]),
      'Modelo': [name1] * len(score_1) + [name2] * len(score_2)
  })

  plt.figure(figsize=(10, 6))
  sns.set_style("whitegrid")

  ax = sns.boxplot(x='Modelo', y='Acurácia', data=df, palette="Set2", width=0.5)

  plt.title(f'Comparação de Acurácia: {name1} vs {name2}', fontsize=14)
  plt.ylabel('Acurácia', fontsize=12)
  plt.xlabel('Modelo', fontsize=12)

  plt.show()

In [ ]:
# Hipotese Nula: modelos são iguais
stat, p_value = friedmanchisquare(knn_accuracy, reglog_accuracy, perceptron_accuracy)
print(p_value)

In [ ]:
#Hipotese nula: modelos tem desempenhos iguais
data = np.column_stack([knn_accuracy, reglog_accuracy, perceptron_accuracy])
posthoc = sp.posthoc_nemenyi_friedman(data)
print(posthoc)

In [ ]:
display(perceptron_accuracy.mean())
display(knn_accuracy.mean())
display(reglog_accuracy.mean())
# display(arvdec_accuracy.mean())

In [ ]:
display(perceptron_f1.mean())
display(knn_f1.mean())
display(reglog_f1.mean())